# Colab Worker C: Track C - Learned Watermark Encoder & Controls
This notebook runs on **Colab Account 3** (T4/V100/A100 GPU).
- Trains Dynamic Band-Allocation Watermarking Network ($k=32$ bits, PSNR=40 dB)
- Runs 100-image sanity overfit check
- Trains 3 Fixed-band Controls: Low (0-2), Mid (3-5), High (6-7)
- Trains Main Model Stage 1 (differentiable noise mixture)
- Once Worker A exports `fitted_g.json`: Trains Main Model Stage 2 with Spectral-Transfer Surrogate (Merge 1)

In [ ]:
# 1. GPU Check & Drive Mount
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
SHARED_DIR = '/content/drive/MyDrive/watermark_internship'
!mkdir -p $SHARED_DIR/checkpoints $SHARED_DIR/results

In [ ]:
# 2. Clone repo & setup dependencies
!rm -rf /content/repo
!git clone https://github.com/ashhyyyy-vis/Internship.git /content/repo
%cd /content/repo
!pip uninstall -y torchao
!pip install -q --no-warn-conflicts diffusers transformers accelerate "compressai>=1.2.4" kornia lpips pandas scipy tqdm

In [ ]:
# 3. Setup Dataset Splits (Drive unpack with automatic download fallback)
import os
!mkdir -p /content/repo/data/processed

# Unpack from Google Drive if processed_splits.zip exists
!if [ -f $SHARED_DIR/processed_splits.zip ]; then \
    echo "Found processed_splits.zip in Google Drive. Extracting..."; \
    unzip -q -o $SHARED_DIR/processed_splits.zip -d /content/repo/data/; \
    unzip -q -o $SHARED_DIR/processed_splits.zip -d /content/repo/data/processed/; \
  fi

# Fallback: if data/fit is still missing, automatically download and prepare in Colab (<30s)
if not os.path.exists('/content/repo/data/fit') and not os.path.exists('/content/repo/data/processed/fit'):
    print("Dataset splits not found in Drive. Preparing dataset directly in Colab...")
    !python data_prep.py
    !cp /content/repo/data/processed_splits.zip $SHARED_DIR/ 2>/dev/null || true

# Mirror directory structure so any path works seamlessly
!mkdir -p /content/repo/data/processed/fit /content/repo/data/processed/test /content/repo/data/processed/train
!cp -rn /content/repo/data/fit/* /content/repo/data/processed/fit/ 2>/dev/null || true
!cp -rn /content/repo/data/test/* /content/repo/data/processed/test/ 2>/dev/null || true
!cp -rn /content/repo/data/train/* /content/repo/data/processed/train/ 2>/dev/null || true

n_fit = len(os.listdir('/content/repo/data/processed/fit')) if os.path.exists('/content/repo/data/processed/fit') else 0
print(f"✓ Dataset ready! Found {n_fit} images available for training.")

In [ ]:
# 4. Step 1: Sanity Overfit Test (100 images, identity noise)
!python -m track_c.train \
    --manifest data/manifests/fit_manifest.json \
    --images_dir data/processed/fit \
    --overfit_100 \
    --epochs 10 \
    --save_dir checkpoints/overfit_check
print('✓ Sanity overfit test passed!')

In [ ]:
# 5. Step 2: Train 3 Fixed-band Controls (Low, Mid, High)
!python -m track_c.train --fixed_regime low --epochs 25 --save_dir checkpoints/control_low
!python -m track_c.train --fixed_regime mid --epochs 25 --save_dir checkpoints/control_mid
!python -m track_c.train --fixed_regime high --epochs 25 --save_dir checkpoints/control_high

!cp -r checkpoints/control_* $SHARED_DIR/checkpoints/
print('✓ Fixed controls trained and copied to Drive!')

In [ ]:
# 6. Step 3: Train Main Model Stage 1 (without surrogate)
!python -m track_c.train --epochs 30 --save_dir checkpoints/main_stage1
!cp -r checkpoints/main_stage1 $SHARED_DIR/checkpoints/
print('✓ Stage 1 main model trained and copied to Drive!')

In [ ]:
# 7. Step 4 (MERGE 1): Train Main Model Stage 2 with Surrogate from Worker A
# Automatic polling loop waits for Worker A to export fitted_g.json to Drive
import os, time
fitted_path = f'{SHARED_DIR}/results/fitted_g.json'
print(f'Checking for Worker A output at: {fitted_path}')

while not os.path.exists(fitted_path):
    print('Waiting for Worker A to export fitted_g.json (polling every 30s)...')
    time.sleep(30)

print('✓ Found fitted_g.json from Worker A! Commencing Stage 2 training...')
!python -m track_c.train \
    --surrogate_json $SHARED_DIR/results/fitted_g.json \
    --epochs 30 \
    --save_dir checkpoints/main_stage2_surrogate

!cp -r checkpoints/main_stage2_surrogate $SHARED_DIR/checkpoints/
print('✓ All Track C models trained successfully!')